# Lab 21 — Images as Matrices

## Pixels, filters, convolution, edges, and vision

This lab is a full computational companion to Chapter 21. The goal is not only to practice commands, but to learn how computer vision begins from linear algebra.

You will represent images as matrices, modify them as arrays, compare them as vectors, apply filters, detect edges, downsample images, and connect these ideas to convolutional neural networks.

**Main theme:** an image is both a structured matrix and a high-dimensional vector.

## 0. Imports and plotting helpers

We use only standard scientific Python tools: NumPy and Matplotlib. Everything is synthetic, so the notebook runs without external image files.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(21)

def show_gray(A, title="", size=4, vmin=0, vmax=1):
    plt.figure(figsize=(size, size))
    plt.imshow(A, cmap="gray", vmin=vmin, vmax=vmax)
    plt.title(title)
    plt.axis("off")
    plt.show()

def show_many(images, titles, cmap="gray", vmin=0, vmax=1, figsize=None):
    k = len(images)
    if figsize is None:
        figsize = (3.2*k, 3.2)
    fig, axes = plt.subplots(1, k, figsize=figsize)
    if k == 1:
        axes = [axes]
    for ax, img, title in zip(axes, images, titles):
        ax.imshow(img, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_title(title)
        ax.axis("off")
    plt.show()

def clip01(A):
    return np.clip(A, 0, 1)

## 1. A tiny image is a matrix

A grayscale image stores one brightness number at each pixel. The following matrix creates a tiny image.

In [ ]:
A = np.array([
    [0, 0, 0, 0, 0, 0],
    [0, 1, 1, 0, 1, 0],
    [0, 1, 1, 0, 1, 0],
    [0, 0, 0, 0, 1, 0],
    [0, 1, 1, 1, 1, 0],
    [0, 0, 0, 0, 0, 0]
], dtype=float)

print(A)
show_gray(A, "A tiny grayscale image", size=3)

### Student task

Modify the matrix above to create your initials, a diagonal line, or a simple icon. Then display it.

In [ ]:
# TODO: Create your own tiny image matrix.
my_image = A.copy()
# Example: uncomment and edit the next line
# my_image[0, :] = 1
show_gray(my_image, "My edited tiny image", size=3)

## 2. Image coordinates: row and column indexing

In Python, `A[i, j]` means row `i`, column `j`. Rows increase downward on the screen.

In [ ]:
B = np.zeros((10, 10))
B[2, 7] = 1.0
B[8, 1] = 0.7

plt.figure(figsize=(5, 5))
plt.imshow(B, cmap="gray", vmin=0, vmax=1)
plt.xticks(range(10))
plt.yticks(range(10))
plt.grid(color="red", linewidth=0.5)
plt.title("Pixel coordinates: row first, column second")
plt.show()

## 3. Creating synthetic images

Synthetic images help us test ideas because we know the exact hidden structure.

In [ ]:
n = 160
img = np.zeros((n, n))

# Rectangle
img[30:115, 25:80] = 0.65

# Circle
rr, cc = np.ogrid[:n, :n]
circle = (rr - 95)**2 + (cc - 110)**2 <= 30**2
img[circle] = 1.0

# Soft background gradient
x = np.linspace(0, 1, n)
img += 0.15 * x[None, :]
img = clip01(img)

show_gray(img, "Synthetic scene: rectangle, circle, gradient", size=5)
print("Image shape:", img.shape)
print("Number of pixels:", img.size)

## 4. Pixelwise operations: brightness, contrast, inversion, gamma

Pixelwise operations apply the same function to every pixel.

In [ ]:
bright = clip01(img + 0.25)
contrast = clip01(0.5 + 1.8*(img - 0.5))
inverted = 1 - img
gamma = img**0.45

show_many(
    [img, bright, contrast, inverted, gamma],
    ["original", "brighter", "higher contrast", "inverted", "gamma corrected"],
    figsize=(15, 3)
)

### Reflection

Which operations preserve object boundaries? Which operations change the relative differences between dark and bright regions?

## 5. Thresholding: from image to decision

Thresholding converts brightness into a binary decision: foreground or background.

In [ ]:
thresholds = [0.25, 0.45, 0.65, 0.85]
binaries = [(img >= t).astype(float) for t in thresholds]
show_many(binaries, [f"t = {t}" for t in thresholds], figsize=(12, 3))

### Student task

Try several thresholds. Which threshold best separates the circle and rectangle from the background gradient?

## 6. Flattening: an image as a vector

A $160 \times 160$ image is a vector in $\mathbb{R}^{25600}$ after flattening.

In [ ]:
v = img.ravel()
print("Matrix shape:", img.shape)
print("Vector shape:", v.shape)
print("First 15 entries of the flattened image:")
print(v[:15])

# Reconstruct the image from the vector
img2 = v.reshape(img.shape)
print("Reconstruction error:", np.linalg.norm(img - img2))

## 7. Image distance

If two images have the same size, their Frobenius distance is the Euclidean distance between their flattened vectors.

In [ ]:
shifted = np.roll(img, shift=8, axis=1)
noisy = clip01(img + rng.normal(0, 0.12, img.shape))
blur_like = clip01(0.7*img + 0.3*np.mean(img))

images = [img, shifted, noisy, blur_like]
titles = ["original", "shifted", "noisy", "lower contrast"]
show_many(images, titles, figsize=(12, 3))

for title, X in zip(titles[1:], images[1:]):
    dist = np.linalg.norm(img - X)
    rel = dist / np.linalg.norm(img)
    print(f"Distance original to {title:14s}: {dist:8.3f}   relative: {rel:6.3f}")

Notice: pixel distance may say a shifted image is quite far away even though humans still recognize the same objects.

## 8. Color images as stacks of matrices

An RGB image has shape `(height, width, 3)`. Each pixel stores a vector `[red, green, blue]`.

In [ ]:
H, W = 120, 180
color = np.zeros((H, W, 3))

# Red horizontal gradient
color[:, :, 0] = np.linspace(0, 1, W)[None, :]
# Green vertical gradient
color[:, :, 1] = np.linspace(0, 1, H)[:, None]
# Blue circular region
rr, cc = np.ogrid[:H, :W]
mask = (rr - 65)**2 + (cc - 115)**2 <= 35**2
color[mask, 2] = 1.0

plt.figure(figsize=(7, 4))
plt.imshow(color)
plt.title("Color image: three matrices stacked together")
plt.axis("off")
plt.show()

show_many([color[:,:,0], color[:,:,1], color[:,:,2]], ["red channel", "green channel", "blue channel"], figsize=(10,3))
print("Color image shape:", color.shape)

## 9. Implementing convolution/filtering

A filter is a local linear operation. It replaces each pixel by a weighted sum of nearby pixels.

In [ ]:
def apply_filter(A, K, padding="edge"):
    A = np.asarray(A, dtype=float)
    K = np.asarray(K, dtype=float)
    kh, kw = K.shape
    ph, pw = kh//2, kw//2
    P = np.pad(A, ((ph, ph), (pw, pw)), mode=padding)
    Y = np.zeros_like(A, dtype=float)
    for i in range(A.shape[0]):
        for j in range(A.shape[1]):
            patch = P[i:i+kh, j:j+kw]
            Y[i, j] = np.sum(patch * K)
    return Y

blur3 = np.ones((3, 3)) / 9
blur7 = np.ones((7, 7)) / 49

blurred3 = apply_filter(img, blur3)
blurred7 = apply_filter(img, blur7)
show_many([img, blurred3, blurred7], ["original", "3x3 blur", "7x7 blur"], figsize=(10,3))

## 10. Edge detection with Sobel filters

Edges are places of strong local change. Sobel filters approximate horizontal and vertical derivatives.

In [ ]:
Kx = np.array([[-1, 0, 1],
               [-2, 0, 2],
               [-1, 0, 1]], dtype=float)
Ky = np.array([[-1, -2, -1],
               [ 0,  0,  0],
               [ 1,  2,  1]], dtype=float)

Ex = apply_filter(img, Kx)
Ey = apply_filter(img, Ky)
Emag = np.sqrt(Ex**2 + Ey**2)
Emag = Emag / Emag.max()

show_many([img, np.abs(Ex), np.abs(Ey), Emag], ["original", "vertical change", "horizontal change", "edge magnitude"], figsize=(13,3))

### Student task

Change the input image. Add a diagonal line, a second circle, or random noise. How does the edge image change?

## 11. Sharpening

A sharpening kernel emphasizes local contrast.

In [ ]:
sharp_kernel = np.array([[0, -1, 0],
                         [-1, 5, -1],
                         [0, -1, 0]], dtype=float)
soft = apply_filter(img, blur7)
sharp = clip01(apply_filter(soft, sharp_kernel))
show_many([img, soft, sharp], ["original", "blurred", "sharpened"], figsize=(10,3))

## 12. Denoising by local averaging

Blurring can remove noise, but too much blur removes details.

In [ ]:
noisy = clip01(img + rng.normal(0, 0.18, img.shape))
den3 = apply_filter(noisy, blur3)
den7 = apply_filter(noisy, blur7)

show_many([img, noisy, den3, den7], ["clean", "noisy", "3x3 denoise", "7x7 denoise"], figsize=(13,3))

for name, X in [("noisy", noisy), ("3x3", den3), ("7x7", den7)]:
    print(f"Error for {name:5s}: {np.linalg.norm(img - X):.3f}")

## 13. Pooling and downsampling

Pooling reduces resolution. Average pooling keeps coarse brightness; max pooling keeps strong activations.

In [ ]:
def avg_pool_2x2(A):
    m, n = A.shape
    A = A[:m//2*2, :n//2*2]
    return A.reshape(m//2, 2, n//2, 2).mean(axis=(1,3))

def max_pool_2x2(A):
    m, n = A.shape
    A = A[:m//2*2, :n//2*2]
    return A.reshape(m//2, 2, n//2, 2).max(axis=(1,3))

avg1 = avg_pool_2x2(img)
avg2 = avg_pool_2x2(avg1)
max1 = max_pool_2x2(img)

show_many([img, avg1, avg2, max1], [f"original {img.shape}", f"avg pool {avg1.shape}", f"avg twice {avg2.shape}", f"max pool {max1.shape}"], figsize=(13,3))

## 14. Filters as feature detectors

A convolutional neural network learns filters from data. Here we manually create a small bank of filters and examine their feature maps.

In [ ]:
filters = {
    "vertical edge": Kx,
    "horizontal edge": Ky,
    "blur": blur3,
    "sharpen": sharp_kernel,
    "center-surround": np.array([[-1,-1,-1],[-1,8,-1],[-1,-1,-1]], dtype=float)
}

feature_maps = []
labels = []
for name, K in filters.items():
    F = apply_filter(img, K)
    if name not in ["blur", "sharpen"]:
        F = np.abs(F)
        F = F / (F.max() + 1e-12)
    else:
        F = clip01(F)
    feature_maps.append(F)
    labels.append(name)

show_many(feature_maps, labels, figsize=(15,3))

## 15. A mini image-classification idea

We can classify simple synthetic images by using features. Here, the feature is the average brightness in the left half and right half.

In [ ]:
def make_left_right_sample(kind, n=32, noise=0.15):
    X = np.zeros((n,n))
    if kind == "left":
        X[:, :n//2] = 1
    elif kind == "right":
        X[:, n//2:] = 1
    elif kind == "center":
        X[:, n//4:3*n//4] = 1
    X = clip01(X + rng.normal(0, noise, X.shape))
    return X

def lr_features(X):
    n = X.shape[1]
    return np.array([X[:, :n//2].mean(), X[:, n//2:].mean()])

samples = [make_left_right_sample("left"), make_left_right_sample("right"), make_left_right_sample("center")]
features = np.array([lr_features(X) for X in samples])
show_many(samples, ["left bright", "right bright", "center bright"], figsize=(9,3))
print("Features [left_mean, right_mean]:")
print(features)

plt.figure(figsize=(5,4))
plt.scatter(features[:,0], features[:,1], s=100)
for label, (a,b) in zip(["left", "right", "center"], features):
    plt.text(a+0.01, b+0.01, label)
plt.xlabel("left half average")
plt.ylabel("right half average")
plt.title("Images become feature vectors")
plt.grid(True, alpha=0.3)
plt.show()

## 16. SVD compression preview

Chapter 17 studied image compression with SVD. Here we revisit the idea briefly: a matrix image can be approximated by a low-rank matrix.

In [ ]:
U, s, Vt = np.linalg.svd(img, full_matrices=False)

def svd_reconstruct(k):
    return (U[:, :k] * s[:k]) @ Vt[:k, :]

ks = [1, 3, 8, 20, 50]
recons = [clip01(svd_reconstruct(k)) for k in ks]
show_many(recons, [f"rank {k}" for k in ks], figsize=(15,3))

energy = np.cumsum(s**2) / np.sum(s**2)
plt.figure(figsize=(6,4))
plt.plot(np.arange(1, len(s)+1), energy)
plt.xlim(1, 60)
plt.ylim(0, 1.02)
plt.xlabel("number of singular values kept")
plt.ylabel("energy captured")
plt.title("SVD energy curve")
plt.grid(True, alpha=0.3)
plt.show()

## 17. Final project: Build your own mini vision pipeline

Choose one synthetic image. Your pipeline should include:

1. Create a clean image.
2. Add noise.
3. Apply a denoising filter.
4. Detect edges.
5. Threshold the edge map.
6. Compute at least one quantitative measure, such as reconstruction error or edge-pixel count.
7. Write a short explanation of what each step means in linear algebra language.

In [ ]:
# Starter template for your project
n = 128
project = np.zeros((n,n))
project[20:95, 30:60] = 0.8
rr, cc = np.ogrid[:n, :n]
project[(rr-70)**2 + (cc-85)**2 <= 22**2] = 1.0

project_noisy = clip01(project + rng.normal(0, 0.16, project.shape))
project_denoised = apply_filter(project_noisy, np.ones((5,5))/25)
px = apply_filter(project_denoised, Kx)
py = apply_filter(project_denoised, Ky)
project_edges = np.sqrt(px**2 + py**2)
project_edges = project_edges / project_edges.max()
project_edge_mask = (project_edges > 0.25).astype(float)

show_many(
    [project, project_noisy, project_denoised, project_edges, project_edge_mask],
    ["clean", "noisy", "denoised", "edges", "edge mask"],
    figsize=(15,3)
)

print("Noisy error:", np.linalg.norm(project - project_noisy))
print("Denoised error:", np.linalg.norm(project - project_denoised))
print("Number of edge pixels:", int(project_edge_mask.sum()))

## 18. Summary

In this lab, you saw that images are a natural home for linear algebra:

- An image is a matrix.
- A color image is a stack of matrices.
- Flattening an image produces a high-dimensional vector.
- Pixelwise operations are functions on entries.
- Filters are local weighted sums.
- Convolution is repeated local dot product.
- Edge detection measures local change.
- Pooling and compression deliberately lose information.
- Feature maps turn images into new representations.

The next chapters use the same philosophy for text, neural networks, recommendation systems, and the grammar of AI.